In [2]:
%load_ext autoreload
%autoreload 2

import sys, logging, importlib

# Ensure your scripts dir is importable
sys.path.append("/global/cfs/cdirs/m4958/usr/danieltm/ColliderML/software/colliderml_dev/scripts/postprocessing")


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uproot
import sys
import seaborn as sns
from tqdm import tqdm
import networkx as nx
import matplotlib.cm as cm
import h5py
import pyarrow as pa
import pyarrow.parquet as pq
import logging
import awkward as ak
import uproot
import time

import atlasify as atl
from particle import Particle
atl.ATLAS = "ColliderML"

sys.path.append("/global/cfs/cdirs/m4958/usr/danieltm/ColliderML/software/OtherLibraries/pyedm4hep")
from pyedm4hep import EDM4hepEvent, EDM4hepEventBatch, utils

from pathlib import Path
import logging

# Reuse postprocessing helpers
sys.path.append("/global/cfs/cdirs/m4958/usr/danieltm/ColliderML/software/colliderml_dev/scripts/postprocessing")
from convert_particles import build_particles_df_with_parents_and_vertex, write_particles_with_selection
from convert_digihits import process_event_for_digihits, write_digihits_with_selection
from convert_calorimeter import write_calohits_with_selection, process_calohits_batch
from utils.path_utils import get_run_paths, make_dir
from utils.track_utils import load_root_file, load_track_summary, create_particle_barcode_map

from convert_all import convert_all


## Roadmap

1. Load trackstates root file from v1
2. Load tracks_ambi.csv file from v1
3. Load trackstates root file from v2
4. Check that all info in v2 is present in v1

In [3]:
included_particles_columns = ['event_id',
 'particle_hash',
 'particle_type',
 'process',
 'vx',
 'vy',
 'vz',
 'vt',
 'px',
 'py',
 'pz',
 'm',
 'q',
 'eta',
 'phi',
 'pt',
 'p',
 'q_over_p',
 'theta',
 'vertex_primary',
 'vertex_secondary',
 'particle',
 'generation',
 'sub_particle',
 'perigee_d0',
 'perigee_z0',
 'perigee_phi',
 'perigee_theta',
 'perigee_q_over_p',
 'perigee_p',
 'perigee_px',
 'perigee_py',
 'perigee_pz',
 'perigee_eta',
 'perigee_pt',
 'e_loss',
 'total_x0',
 'total_l0',
 'number_of_hits',
 'outcome']

In [4]:
included_trackstates_columns = ['event_nr',
 'track_nr',
 'nStates',
 'nMeasurements',
 'volume_id',
 'layer_id',
 'module_id',
 'measurement_id',
 'stateType',
 'chi2',
 'pathLength',
 't_x',
 't_y',
 't_z',
 't_r',
 't_dx',
 't_dy',
 't_dz',
 't_eLOC0',
 't_eLOC1',
 't_ePHI',
 't_eTHETA',
 't_eQOP',
 't_eT',
 'dim_hit',
 'l_x_hit',
 'l_y_hit',
 'g_x_hit',
 'g_y_hit',
 'g_z_hit',
 'res_x_hit',
 'res_y_hit',
 'err_x_hit',
 'err_y_hit',
 'pull_x_hit',
 'pull_y_hit',
 'nPredicted',
 'predicted',
 'eLOC0_prt',
 'eLOC1_prt',
 'ePHI_prt',
 'eTHETA_prt',
 'eQOP_prt',
 'eT_prt',
 'res_eLOC0_prt',
 'res_eLOC1_prt',
 'res_ePHI_prt',
 'res_eTHETA_prt',
 'res_eQOP_prt',
 'res_eT_prt',
 'err_eLOC0_prt',
 'err_eLOC1_prt',
 'err_ePHI_prt',
 'err_eTHETA_prt',
 'err_eQOP_prt',
 'err_eT_prt',
 'pull_eLOC0_prt',
 'pull_eLOC1_prt',
 'pull_ePHI_prt',
 'pull_eTHETA_prt',
 'pull_eQOP_prt',
 'pull_eT_prt',
 'g_x_prt',
 'g_y_prt',
 'g_z_prt',
 'px_prt',
 'py_prt',
 'pz_prt',
 'eta_prt',
 'pT_prt',
 'nFiltered',
 'filtered',
 'eLOC0_flt',
 'eLOC1_flt',
 'ePHI_flt',
 'eTHETA_flt',
 'eQOP_flt',
 'eT_flt',
 'res_eLOC0_flt',
 'res_eLOC1_flt',
 'res_ePHI_flt',
 'res_eTHETA_flt',
 'res_eQOP_flt',
 'res_eT_flt',
 'err_eLOC0_flt',
 'err_eLOC1_flt',
 'err_ePHI_flt',
 'err_eTHETA_flt',
 'err_eQOP_flt',
 'err_eT_flt',
 'pull_eLOC0_flt',
 'pull_eLOC1_flt',
 'pull_ePHI_flt',
 'pull_eTHETA_flt',
 'pull_eQOP_flt',
 'pull_eT_flt',
 'g_x_flt',
 'g_y_flt',
 'g_z_flt',
 'px_flt',
 'py_flt',
 'pz_flt',
 'eta_flt',
 'pT_flt',
 'nSmoothed',
 'smoothed',
 'eLOC0_smt',
 'eLOC1_smt',
 'ePHI_smt',
 'eTHETA_smt',
 'eQOP_smt',
 'eT_smt',
 'res_eLOC0_smt',
 'res_eLOC1_smt',
 'res_ePHI_smt',
 'res_eTHETA_smt',
 'res_eQOP_smt',
 'res_eT_smt',
 'err_eLOC0_smt',
 'err_eLOC1_smt',
 'err_ePHI_smt',
 'err_eTHETA_smt',
 'err_eQOP_smt',
 'err_eT_smt',
 'pull_eLOC0_smt',
 'pull_eLOC1_smt',
 'pull_ePHI_smt',
 'pull_eTHETA_smt',
 'pull_eQOP_smt',
 'pull_eT_smt',
 'g_x_smt',
 'g_y_smt',
 'g_z_smt',
 'px_smt',
 'py_smt',
 'pz_smt',
 'eta_smt',
 'pT_smt',
 'nUnbiased',
 'unbiased',
 'eLOC0_ubs',
 'eLOC1_ubs',
 'ePHI_ubs',
 'eTHETA_ubs',
 'eQOP_ubs',
 'eT_ubs',
 'res_eLOC0_ubs',
 'res_eLOC1_ubs',
 'res_ePHI_ubs',
 'res_eTHETA_ubs',
 'res_eQOP_ubs',
 'res_eT_ubs',
 'err_eLOC0_ubs',
 'err_eLOC1_ubs',
 'err_ePHI_ubs',
 'err_eTHETA_ubs',
 'err_eQOP_ubs',
 'err_eT_ubs',
 'pull_eLOC0_ubs',
 'pull_eLOC1_ubs',
 'pull_ePHI_ubs',
 'pull_eTHETA_ubs',
 'pull_eQOP_ubs',
 'pull_eT_ubs',
 'g_x_ubs',
 'g_y_ubs',
 'g_z_ubs',
 'px_ubs',
 'py_ubs',
 'pz_ubs',
 'eta_ubs',
 'pT_ubs']

### V1

In [5]:
## Load v1

trackstates_v1 = "/global/cfs/cdirs/m4958/data/ColliderML/simulation/hard_scatter/ttbar/v1/runs/0/trackstates_ambi.root"
tracksummary_v1 = "/global/cfs/cdirs/m4958/data/ColliderML/simulation/hard_scatter/ttbar/v1/runs/0/tracksummary_ambi.root"
tracks_csv_v1 = "/global/cfs/cdirs/m4958/data/ColliderML/simulation/hard_scatter/ttbar/v1/runs/0/event000000000-tracks_ambi.csv"
particles_v1 = "/global/cfs/cdirs/m4958/data/ColliderML/simulation/hard_scatter/ttbar/v1/runs/0/particles.root"

In [6]:
track_summary_file = uproot.open(tracksummary_v1)
track_summary_file["tracksummary"].keys()


['event_nr',
 'track_nr',
 'nStates',
 'nMeasurements',
 'nOutliers',
 'nHoles',
 'nSharedHits',
 'chi2Sum',
 'NDF',
 'measurementChi2',
 'outlierChi2',
 'measurementVolume',
 'measurementLayer',
 'outlierVolume',
 'outlierLayer',
 'nMajorityHits',
 'majorityParticleId',
 'trackClassification',
 't_charge',
 't_time',
 't_vx',
 't_vy',
 't_vz',
 't_px',
 't_py',
 't_pz',
 't_theta',
 't_phi',
 't_eta',
 't_p',
 't_pT',
 't_d0',
 't_z0',
 't_prodR',
 'hasFittedParams',
 'eLOC0_fit',
 'eLOC1_fit',
 'ePHI_fit',
 'eTHETA_fit',
 'eQOP_fit',
 'eT_fit',
 'err_eLOC0_fit',
 'err_eLOC1_fit',
 'err_ePHI_fit',
 'err_eTHETA_fit',
 'err_eQOP_fit',
 'err_eT_fit',
 'res_eLOC0_fit',
 'res_eLOC1_fit',
 'res_ePHI_fit',
 'res_eTHETA_fit',
 'res_eQOP_fit',
 'res_eT_fit',
 'pull_eLOC0_fit',
 'pull_eLOC1_fit',
 'pull_ePHI_fit',
 'pull_eTHETA_fit',
 'pull_eQOP_fit',
 'pull_eT_fit',
 'cov_eLOC0_eLOC0',
 'cov_eLOC0_eLOC1',
 'cov_eLOC0_ePHI',
 'cov_eLOC0_eTHETA',
 'cov_eLOC0_eQOP',
 'cov_eLOC0_eT',
 'cov_eLOC1_e

In [7]:
track_fitting_file = uproot.open(trackstates_v1)
track_fitting_file["trackstates"].keys()

['event_nr',
 'track_nr',
 'nStates',
 'nMeasurements',
 'volume_id',
 'layer_id',
 'module_id',
 'stateType',
 'chi2',
 'pathLength',
 't_x',
 't_y',
 't_z',
 't_r',
 't_dx',
 't_dy',
 't_dz',
 't_eLOC0',
 't_eLOC1',
 't_ePHI',
 't_eTHETA',
 't_eQOP',
 't_eT',
 'dim_hit',
 'l_x_hit',
 'l_y_hit',
 'g_x_hit',
 'g_y_hit',
 'g_z_hit',
 'res_x_hit',
 'res_y_hit',
 'err_x_hit',
 'err_y_hit',
 'pull_x_hit',
 'pull_y_hit',
 'nPredicted',
 'predicted',
 'eLOC0_prt',
 'eLOC1_prt',
 'ePHI_prt',
 'eTHETA_prt',
 'eQOP_prt',
 'eT_prt',
 'res_eLOC0_prt',
 'res_eLOC1_prt',
 'res_ePHI_prt',
 'res_eTHETA_prt',
 'res_eQOP_prt',
 'res_eT_prt',
 'err_eLOC0_prt',
 'err_eLOC1_prt',
 'err_ePHI_prt',
 'err_eTHETA_prt',
 'err_eQOP_prt',
 'err_eT_prt',
 'pull_eLOC0_prt',
 'pull_eLOC1_prt',
 'pull_ePHI_prt',
 'pull_eTHETA_prt',
 'pull_eQOP_prt',
 'pull_eT_prt',
 'g_x_prt',
 'g_y_prt',
 'g_z_prt',
 'px_prt',
 'py_prt',
 'pz_prt',
 'eta_prt',
 'pT_prt',
 'nFiltered',
 'filtered',
 'eLOC0_flt',
 'eLOC1_flt',
 'ePHI

In [8]:
particles_file = uproot.open(particles_v1)
particles_file["particles"].keys()


['event_id',
 'particle_hash',
 'particle_type',
 'process',
 'vx',
 'vy',
 'vz',
 'vt',
 'px',
 'py',
 'pz',
 'm',
 'q',
 'eta',
 'phi',
 'pt',
 'p',
 'q_over_p',
 'theta',
 'vertex_primary',
 'vertex_secondary',
 'particle',
 'generation',
 'sub_particle',
 'e_loss',
 'total_x0',
 'total_l0',
 'number_of_hits',
 'outcome']

In [9]:
included_tracksummary_columns = ["event_nr", "track_nr", "eLOC0_fit", "eLOC1_fit", "ePHI_fit", "eTHETA_fit", "eQOP_fit", "eT_fit", "t_d0", "t_z0", "t_phi", "t_theta", "t_charge", "t_p", "t_pT", "t_time", "hasFittedParams", 'nMajorityHits', 'majorityParticleId', 'trackClassification']
track_summary_df = load_root_file(str(tracksummary_v1), included_columns=included_tracksummary_columns, events=(0,1))


In [10]:
track_states_df = load_root_file(str(trackstates_v1), included_columns=included_trackstates_columns, events=(0,1))

In [11]:
track_summary_df

event_id  track_nr  nMajorityHits  \
entry subentry subsubentry                                      
0     0        0                   0         0     4294967295   
               1                   0         0     4294967295   
               2                   0         0     4294967295   
               3                   0         0     4294967295   
               4                   0         0     4294967295   
...                              ...       ...            ...   
      17       0                   0        17             14   
               1                   0        17             14   
               2                   0        17             14   
               3                   0        17             14   
               4                   0        17             14   

                            majorityParticleId  trackClassification  \
entry subentry subsubentry                                            
0     0        0                             0                    0   
               1                             0                    0   
               2                             0                    0   
               3                             0                    0   
               4                             0                    0   
...                                        ...                  ...   
      17       0                             1                    1   
               1                             0                    1   
               2                             0                    1   
               3                             1                    1   
               4                           159                    1   

                              t_charge      t_time   t_theta     t_phi  \
entry subentry subsubentry                                               
0     0        0            2147483647         NaN       NaN       NaN   
               1            2147483647         NaN       NaN       NaN   
               2            2147483647         NaN       NaN       NaN   
               3            2147483647         NaN       NaN       NaN   
               4            2147483647         NaN       NaN       NaN   
...                                ...         ...       ...       ...   
      17       0                     1  132.644058  2.257941  3.061142   
               1                     1  132.644058  2.257941  3.061142   
               2                     1  132.644058  2.257941  3.061142   
               3                     1  132.644058  2.257941  3.061142   
               4                     1  132.644058  2.257941  3.061142   

                                 t_p      t_pT      t_d0      t_z0  \
entry subentry subsubentry                                           
0     0        0                 NaN       NaN       NaN       NaN   
               1                 NaN       NaN       NaN       NaN   
               2                 NaN       NaN       NaN       NaN   
               3                 NaN       NaN       NaN       NaN   
               4                 NaN       NaN       NaN       NaN   
...                              ...       ...       ...       ...   
      17       0            1.317205  1.018279  0.002946  196.0625   
               1            1.317205  1.018279  0.002946  196.0625   
               2            1.317205  1.018279  0.002946  196.0625   
               3            1.317205  1.018279  0.002946  196.0625   
               4            1.317205  1.018279  0.002946  196.0625   

                            hasFittedParams  eLOC0_fit   eLOC1_fit  ePHI_fit  \
entry subentry subsubentry                                                     
0     0        0                       True   0.032012  196.103989 -2.826225   
               1                       True   0.032012  196.103989 -2.826225   
               2                       True   0.032012  196.1039

In [12]:
track_states_df

event_id  track_nr  nStates  nMeasurements  volume_id  \
entry subentry                                                          
0     0                0         0       12              6         24   
      1                0         0       12              6         24   
      2                0         0       12              6         20   
      3                0         0       12              6         17   
      4                0         0       12              6         17   
...                  ...       ...      ...            ...        ...   
17    9                0        17       26             14         24   
      10               0        17       26             14         24   
      11               0        17       26             14         24   
      12               0        17       26             14         24   
      13               0        17       26             14         24   

                layer_id  module_id  stateType      chi2  pathLength  ...  \
entry subentry                                                        ...   
0     0                2        318          0  2.177661  276.230988  ...   
      1                2          0          3  0.000000  257.536407  ...   
      2                2          0          3  0.000000  204.266479  ...   
      3                8          0          3  0.000000  172.484253  ...   
      4                8        559          0  0.765201  166.326645  ...   
...                  ...        ...        ...       ...         ...  ...   
17    9                8          0          3  0.000000  805.035217  ...   
      10               6        629          0  0.950681  609.939453  ...   
      11               6          0          3  0.000000  591.544678  ...   
      12               4        420          0  0.297325  423.390991  ...   
      13               4          0          3  0.000000  407.943695  ...   

                pull_eQOP_ubs  pull_eT_ubs     g_x_ubs     g_y_ubs  \
entry subentry                                                       
0     0            -92.672974    -0.057337 -244.565613  -96.698662   
      1                   NaN          NaN         NaN         NaN   
      2                   NaN          NaN         NaN         NaN   
      3                   NaN          NaN         NaN         NaN   
      4           -130.360779    -0.057287 -160.642487  -59.637707   
...                       ...          ...         ...         ...   
17    9                   NaN          NaN         NaN         NaN   
      10            -1.053508    -0.058528 -476.730469  153.513199   
      11                  NaN          NaN         NaN         NaN   
      12            -1.300038    -0.058472 -348.116394   87.380173   
      13                  NaN          NaN         NaN         NaN   

                   g_z_ubs    px_ubs    py_ubs    pz_ubs   eta_ubs    pT_ubs  
entry subentry                                                                
0     0          22.577421 -1.764858 -0.824744 -1.284777 -0.619186  1.948057  
      1                NaN       NaN       NaN       NaN       NaN       NaN  
      2                NaN       NaN       NaN       NaN       NaN       NaN  
      3                NaN       NaN       NaN       NaN       NaN       NaN  
      4          83.088371 -1.765223 -0.736593 -1.261472 -0.619180  1.912742  
...                    ...       ...       ...       ...       ...       ...  
17    9                NaN       NaN       NaN       NaN       NaN       NaN  
      10       -216.119705 -0.847687  0.508251 -0.805037 -0.743953  0.988378  
      11               NaN       NaN       NaN       NaN       NaN       NaN  
      12        -98.388458 -0.908134  0.395454 -0.806606 -0.743828  0.990501  
      13               NaN       NaN       NaN       NaN       NaN       NaN  

[228 rows x 171 columns]

In [13]:
print(track_states_df.columns)

Index(['event_id', 'track_nr', 'nStates', 'nMeasurements', 'volume_id',
       'layer_id', 'module_id', 'stateType', 'chi2', 'pathLength',
       ...
       'pull_eQOP_ubs', 'pull_eT_ubs', 'g_x_ubs', 'g_y_ubs', 'g_z_ubs',
       'px_ubs', 'py_ubs', 'pz_ubs', 'eta_ubs', 'pT_ubs'],
      dtype='object', length=171)


In [14]:
tracks_csv = pd.read_csv(tracks_csv_v1)

In [15]:
tracks_csv

,track_id,seed_id,particleId,nStates,nMajorityHits,nMeasurements,nOutliers,nHoles,nSharedHits,chi2,ndf,chi2/ndf,pT,eta,phi,truthMatchProbability,good/duplicate/fake,Measurements_ID
0,16,29,vp=1|vs=0|p=0|g=1|sp=150,27,15,15,0,0,0,15.5426,26,0.597792,1.55393,-0.418693,1.701410,1.00000,good,"[124,251,255,279,401,406,732,826,938,1027,1026..."
1,15,28,vp=1|vs=0|p=0|g=1|sp=154,25,13,13,0,0,0,30.9455,22,1.406610,2.02290,-0.196890,1.584390,1.00000,good,"[123,250,270,397,731,827,821,937,1021,1320,132..."
2,1,1,vp=1|vs=0|p=0|g=0|sp=30,24,11,11,0,0,0,10.6233,18,0.590183,1.28720,1.489750,-2.705780,1.00000,good,"[145,175,306,759,867,1079,1109,1552,1553,1574,..."
3,2,2,vp=1|vs=0|p=0|g=0|sp=36,16,8,8,0,0,0,14.2295,16,0.889342,4.72857,2.809390,-2.062620,1.00000,good,"[79,460,477,496,518,565,600,641,]"
4,3,4,vp=1|vs=0|p=0|g=1|sp=110,25,15,15,0,0,0,27.0792,30,0.902639,1.22903,2.940130,-1.994130,1.00000,good,"[80,461,459,478,476,498,495,525,517,554,548,60..."
5,4,5,vp=1|vs=0|p=0|g=0|sp=13,28,13,13,0,0,0,20.9450,22,0.952046,18.16820,1.630160,-1.521510,1.00000,good,"[84,322,323,774,886,885,1081,1110,1136,1568,15..."
6,5,7,vp=1|vs=0|p=0|g=1|sp=10,26,13,11,3,0,0,26.5955,20,1.329770,1.43955,-0.360456,0.656637,1.18182,good,"[71,102,216,332,452,453,791,895,979,1058,1363,..."
7,6,8,vp=1|vs=18|p=0|g=2|sp=11,25,13,13,0,0,0,17.6611,22,0.802778,5.20938,-0.525360,0.779719,1.00000,good,"[105,230,341,455,387,793,902,1008,1009,1390,13..."
8,7,9,vp=1|vs=0|p=0|g=3|sp=6,26,14,14,0,0,0,23.4133,24,0.975554,3.91352,-0.478465,0.872830,1.00000,good,"[103,228,233,351,385,390,798,901,920,1012,1389..."
9,8,10,vp=1|vs=2|p=0|g=2|sp=9,28,16,16,0,0,0,14.8858,26,0.572530,2.77748,-0.403463,0.817460,1.00000,good,"[101,226,357,384,389,801,797,913,925,1017,1403..."


### V2

In [ ]:
## Load v2

trackstates_v2 = "/global/cfs/cdirs/m4958/data/ColliderML/simulation/hard_scatter/ttbar/v2/runs/0/trackstates_ambi.root"
tracksummary_v2 = "/global/cfs/cdirs/m4958/data/ColliderML/simulation/hard_scatter/ttbar/v2/runs/0/tracksummary_ambi.root"
tracks_csv_v2 = "/global/cfs/cdirs/m4958/data/ColliderML/simulation/hard_scatter/ttbar/v1/runs/0/event000000000-tracks_ambi.csv"
particles_v2 = "/global/cfs/cdirs/m4958/data/ColliderML/simulation/hard_scatter/ttbar/v2/runs/0/particles.root"

In [6]:
track_summary_file = uproot.open(tracksummary_v2)
track_summary_file["tracksummary"].keys()


['event_nr',
 'track_nr',
 'nStates',
 'nMeasurements',
 'nOutliers',
 'nHoles',
 'nSharedHits',
 'chi2Sum',
 'NDF',
 'measurementChi2',
 'outlierChi2',
 'measurementVolume',
 'measurementLayer',
 'outlierVolume',
 'outlierLayer',
 'measurementIDs',
 'nMajorityHits',
 'majorityParticleId',
 'trackClassification',
 't_charge',
 't_time',
 't_vx',
 't_vy',
 't_vz',
 't_px',
 't_py',
 't_pz',
 't_theta',
 't_phi',
 't_eta',
 't_p',
 't_pT',
 't_d0',
 't_z0',
 't_prodR',
 'hasFittedParams',
 'eLOC0_fit',
 'eLOC1_fit',
 'ePHI_fit',
 'eTHETA_fit',
 'eQOP_fit',
 'eT_fit',
 'err_eLOC0_fit',
 'err_eLOC1_fit',
 'err_ePHI_fit',
 'err_eTHETA_fit',
 'err_eQOP_fit',
 'err_eT_fit',
 'res_eLOC0_fit',
 'res_eLOC1_fit',
 'res_ePHI_fit',
 'res_eTHETA_fit',
 'res_eQOP_fit',
 'res_eT_fit',
 'pull_eLOC0_fit',
 'pull_eLOC1_fit',
 'pull_ePHI_fit',
 'pull_eTHETA_fit',
 'pull_eQOP_fit',
 'pull_eT_fit',
 'cov_eLOC0_eLOC0',
 'cov_eLOC0_eLOC1',
 'cov_eLOC0_ePHI',
 'cov_eLOC0_eTHETA',
 'cov_eLOC0_eQOP',
 'cov_eLOC0

In [7]:
track_summary_file["tracksummary"].arrays()

<Array [{event_nr: 0, track_nr: ..., ...}, ...] type='8 * {event_nr: uint32...'>

In [8]:
track_fitting_file = uproot.open(trackstates_v2)
track_fitting_file["trackstates"].keys()

['event_nr',
 'track_nr',
 'nStates',
 'nMeasurements',
 'volume_id',
 'layer_id',
 'module_id',
 'measurement_id',
 'stateType',
 'chi2',
 'pathLength',
 't_x',
 't_y',
 't_z',
 't_r',
 't_dx',
 't_dy',
 't_dz',
 't_eLOC0',
 't_eLOC1',
 't_ePHI',
 't_eTHETA',
 't_eQOP',
 't_eT',
 'dim_hit',
 'l_x_hit',
 'l_y_hit',
 'g_x_hit',
 'g_y_hit',
 'g_z_hit',
 'res_x_hit',
 'res_y_hit',
 'err_x_hit',
 'err_y_hit',
 'pull_x_hit',
 'pull_y_hit',
 'nPredicted',
 'predicted',
 'eLOC0_prt',
 'eLOC1_prt',
 'ePHI_prt',
 'eTHETA_prt',
 'eQOP_prt',
 'eT_prt',
 'res_eLOC0_prt',
 'res_eLOC1_prt',
 'res_ePHI_prt',
 'res_eTHETA_prt',
 'res_eQOP_prt',
 'res_eT_prt',
 'err_eLOC0_prt',
 'err_eLOC1_prt',
 'err_ePHI_prt',
 'err_eTHETA_prt',
 'err_eQOP_prt',
 'err_eT_prt',
 'pull_eLOC0_prt',
 'pull_eLOC1_prt',
 'pull_ePHI_prt',
 'pull_eTHETA_prt',
 'pull_eQOP_prt',
 'pull_eT_prt',
 'g_x_prt',
 'g_y_prt',
 'g_z_prt',
 'px_prt',
 'py_prt',
 'pz_prt',
 'eta_prt',
 'pT_prt',
 'nFiltered',
 'filtered',
 'eLOC0_flt',
 

In [9]:
particles_file = uproot.open(particles_v2)
particles_file["particles"].keys()


['event_id',
 'particle_hash',
 'particle_type',
 'process',
 'vx',
 'vy',
 'vz',
 'vt',
 'px',
 'py',
 'pz',
 'm',
 'q',
 'eta',
 'phi',
 'pt',
 'p',
 'q_over_p',
 'theta',
 'vertex_primary',
 'vertex_secondary',
 'particle',
 'generation',
 'sub_particle',
 'perigee_d0',
 'perigee_z0',
 'perigee_phi',
 'perigee_theta',
 'perigee_q_over_p',
 'perigee_p',
 'perigee_px',
 'perigee_py',
 'perigee_pz',
 'perigee_eta',
 'perigee_pt',
 'e_loss',
 'total_x0',
 'total_l0',
 'number_of_hits',
 'outcome']

In [10]:
included_tracksummary_columns = ["event_nr", "track_nr", "eLOC0_fit", "eLOC1_fit", "ePHI_fit", "eTHETA_fit", "eQOP_fit", "eT_fit", "t_d0", "t_z0", "t_phi", "t_theta", "t_charge", "t_p", "t_pT", "t_time", "hasFittedParams", 'trackClassification', 'nMajorityHits', 'measurementIDs']#'majorityParticleId']#, ]
track_summary_df = load_root_file(str(tracksummary_v2), included_columns=included_tracksummary_columns, events=(0,1))


In [11]:
track_states_df = load_root_file(str(trackstates_v2), included_columns=included_trackstates_columns, events=(0,1))

In [12]:
particles_df = load_root_file(str(particles_v2), included_columns=included_particles_columns, events=(0,1))

In [13]:
(~particles_df.perigee_px.isna()).sum()

np.int64(1537)

In [20]:
(particles_df.perigee_px.isna()).sum()

np.int64(308)

In [14]:
# Given eQOP_fit, calculate fitted pt
# Need to use theta to calculate pt
# pT = |1/eQOP_fit| * sin(theta)  (with theta = eTHETA_fit)
track_summary_df["PT_fit"] = track_summary_df["eQOP_fit"].apply(lambda qop: 0 if qop == 0 else 1.0/abs(qop)) * track_summary_df["eTHETA_fit"].apply(np.sin)

In [15]:
track_summary_df

event_id  track_nr  measurementIDs  nMajorityHits  \
entry subentry subsubentry                                                      
0     0        0                   0         0             758     4294967295   
               1                   0         0             434     4294967295   
               2                   0         0             429     4294967295   
               3                   0         0             307     4294967295   
               4                   0         0             179     4294967295   
...                              ...       ...             ...            ...   
      17       9                   0        17             418             14   
               10                  0        17             299             14   
               11                  0        17             170             14   
               12                  0        17             168             14   
               13                  0        17             141             14   

                            trackClassification    t_charge      t_time  \
entry subentry subsubentry                                                
0     0        0                              0  2147483647         NaN   
               1                              0  2147483647         NaN   
               2                              0  2147483647         NaN   
               3                              0  2147483647         NaN   
               4                              0  2147483647         NaN   
...                                         ...         ...         ...   
      17       9                              1           1  132.644058   
               10                             1           1  132.644058   
               11                             1           1  132.644058   
               12                             1           1  132.644058   
               13                             1           1  132.644058   

                             t_theta     t_phi       t_p  ...      t_d0  \
entry subentry subsubentry                                ...             
0     0        0                 NaN       NaN       NaN  ...       NaN   
               1                 NaN       NaN       NaN  ...       NaN   
               2                 NaN       NaN       NaN  ...       NaN   
               3                 NaN       NaN       NaN  ...       NaN   
               4                 NaN       NaN       NaN  ...       NaN   
...                              ...       ...       ...  ...       ...   
      17       9            2.257941  3.061142  1.317205  ...  0.002946   
               10           2.257941  3.061142  1.317205  ...  0.002946   
               11           2.257941  3.061142  1.317205  ...  0.002946   
               12           2.257941  3.061142  1.317205  ...  0.002946   
               13           2.257941  3.061142  1.317205  ...  0.002946   

                                t_z0  hasFittedParams  eLOC0_fit   eLOC1_fit  \
entry subentry subsubentry                                                     
0     0        0                 NaN             True   0.022656  196.087402   
               1                 NaN             True   0.022656  196.087402   
               2                 NaN             True   0.022656  196.087402   
               3                 NaN             True   0.022656  196.087402   
               4                 NaN             True   0.022656  196.087402   
...                              ...              ...        ...         ...   
      17       9            196.0625             True   0.032160  196.155594   
               10           196.0625             True   0.032160  196.155594   
               11           196.0625             True   0.032160  196.155594   
               12           196.0625             True   0.032160  196.155594   
               13           196.0625             True   0.0321

In [16]:
tracks_csv = pd.read_csv(tracks_csv_v2)

In [17]:
for measurements in tracks_csv[tracks_csv.track_id == 1]["Measurements_ID"]:
    print(measurements)

[145,175,306,759,867,1079,1109,1552,1553,1574,1576,]


In [18]:
track_states_df[track_states_df.track_nr == 1][["track_nr", "measurement_id"]]

track_nr        measurement_id
entry subentry                                
1     0                1                  1576
      1                1                  1574
      2                1  18446744073709551615
      3                1  18446744073709551615
      4                1                  1553
      5                1                  1552
      6                1  18446744073709551615
      7                1  18446744073709551615
      8                1                  1109
      9                1                  1079
      10               1  18446744073709551615

In [19]:
track_summary_df[track_summary_df.track_nr == 1][["track_nr", "measurementIDs"]]

track_nr  measurementIDs
entry subentry subsubentry                          
0     1        0                   1            1576
               1                   1            1574
               2                   1            1553
               3                   1            1552
               4                   1            1109
               5                   1            1079
               6                   1             867
               7                   1             759
               8                   1             306
               9                   1             175
               10                  1             145

## Summaries

In [31]:
# The value 18446744073709551615 is UINT64_MAX (2^64 - 1), which ACTS uses as a sentinel
# to mark states WITHOUT measurements (predicted states, holes, outliers, material interactions, etc.)
# 
# The CSV file (from CsvTrackWriter) only contains measurement states (actual hits),
# while trackstates contains ALL states along the track.
#
# To compare properly, filter trackstates to only measurement states:
UINT64_MAX = 18446744073709551615

# Filter to only states with actual measurements
measurement_states = track_states_df[
    (track_states_df.track_nr == 1) & 
    (track_states_df.measurement_id != UINT64_MAX)
][["track_nr", "measurement_id", "stateType"]]

print("Measurement states from trackstates (filtered):")
print(measurement_states)
print("\nMeasurement IDs from CSV:")
csv_measurements = tracks_csv[tracks_csv.track_id == 1]["Measurements_ID"].iloc[0]
print(csv_measurements)

# Extract measurement IDs from CSV string
import ast
csv_ids = ast.literal_eval(csv_measurements) if isinstance(csv_measurements, str) else csv_measurements
trackstates_ids = measurement_states["measurement_id"].tolist()

print(f"\nCSV measurement IDs: {sorted(csv_ids)}")
print(f"Trackstates measurement IDs: {sorted(trackstates_ids)}")
print(f"\nMatch: {set(csv_ids) == set(trackstates_ids)}")


Measurement states from trackstates (filtered):
                track_nr  measurement_id  stateType
entry subentry                                     
1     0                1            1576          0
      1                1            1574          0
      4                1            1553          0
      5                1            1552          0
      8                1            1109          0
      9                1            1079          0

Measurement IDs from CSV:
[145,175,306,759,867,1079,1109,1552,1553,1574,1576,]

CSV measurement IDs: [145, 175, 306, 759, 867, 1079, 1109, 1552, 1553, 1574, 1576]
Trackstates measurement IDs: [1079, 1109, 1552, 1553, 1574, 1576]

Match: False


In [32]:
# Let's investigate: Where did the measurement IDs go?
UINT64_MAX = 18446744073709551615

# Get all states for track 1
track1_states = track_states_df[track_states_df.track_nr == 1].copy()
print(f"Total states for track 1: {len(track1_states)}")
print(f"States with valid measurement_id: {len(track1_states[track1_states.measurement_id != UINT64_MAX])}")
print(f"States with sentinel measurement_id: {len(track1_states[track1_states.measurement_id == UINT64_MAX])}")

# Get CSV measurement IDs
csv_measurements = tracks_csv[tracks_csv.track_id == 1]["Measurements_ID"].iloc[0]
import ast
csv_ids = ast.literal_eval(csv_measurements) if isinstance(csv_measurements, str) else csv_measurements
print(f"\nCSV measurement IDs: {sorted(csv_ids)}")

# Check if any CSV measurement IDs appear in trackstates
valid_measurement_ids = track1_states[track1_states.measurement_id != UINT64_MAX]["measurement_id"].tolist()
print(f"Valid measurement IDs in trackstates: {sorted(valid_measurement_ids)}")

# Check if CSV IDs appear anywhere in trackstates (even with sentinel)
all_measurement_ids = track1_states["measurement_id"].tolist()
missing_ids = set(csv_ids) - set(valid_measurement_ids)
print(f"\nMissing measurement IDs (in CSV but not in trackstates): {sorted(missing_ids)}")

# Show stateType for states with sentinel values
print("\nState types for states with sentinel measurement_id:")
sentinel_states = track1_states[track1_states.measurement_id == UINT64_MAX][["measurement_id", "stateType", "nMeasurements"]]
print(sentinel_states)


Total states for track 1: 11
States with valid measurement_id: 6
States with sentinel measurement_id: 5

CSV measurement IDs: [145, 175, 306, 759, 867, 1079, 1109, 1552, 1553, 1574, 1576]
Valid measurement IDs in trackstates: [1079, 1109, 1552, 1553, 1574, 1576]

Missing measurement IDs (in CSV but not in trackstates): [145, 175, 306, 759, 867]

State types for states with sentinel measurement_id:
                      measurement_id  stateType  nMeasurements
entry subentry                                                
1     2         18446744073709551615          3             11
      3         18446744073709551615          3             11
      6         18446744073709551615          3             11
      7         18446744073709551615          3             11
      10        18446744073709551615          3             11


In [33]:
# Let's also check the stateType values to understand what types of states we have
print("StateType value meanings:")
print("(Need to check ACTS source, but typically: 0=Measurement, 1=Outlier, 2=Hole, 3=Material, etc.)")
print("\nAll stateType values for track 1:")
print(track1_states[["measurement_id", "stateType", "nMeasurements"]].value_counts(subset=["stateType"]))

# Show full details of all states
print("\nAll states for track 1:")
print(track1_states[["track_nr", "measurement_id", "stateType", "nMeasurements", "volume_id", "layer_id"]].to_string())


StateType value meanings:
(Need to check ACTS source, but typically: 0=Measurement, 1=Outlier, 2=Hole, 3=Material, etc.)

All stateType values for track 1:
stateType
0            6
3            5
Name: count, dtype: int64

All states for track 1:
                track_nr        measurement_id  stateType  nMeasurements  volume_id  layer_id
entry subentry                                                                               
1     0                1                  1576          0             11         30         8
      1                1                  1574          0             11         30         8
      2                1  18446744073709551615          3             11         30         8
      3                1  18446744073709551615          3             11         30         6
      4                1                  1553          0             11         30         6
      5                1                  1552          0             11         30         6
 

In [34]:
# Critical insight: The CSV writer checks MeasurementFlag, while trackstates writer checks hasUncalibratedSourceLink()
# A state can have MeasurementFlag but NO source link (e.g., outliers that were removed)
# OR there could be a bug where source links are lost

# Let's check: Are there states with stateType=0 (Measurement) but measurement_id=UINT64_MAX?
# stateType enum: 0=Measurement, 1=Outlier, 2=Hole, 3=Material, 4=Unknown
print("States with stateType=0 (Measurement) but sentinel measurement_id:")
measurement_type_states = track1_states[
    (track1_states.stateType == 0) & 
    (track1_states.measurement_id == UINT64_MAX)
]
print(measurement_type_states[["measurement_id", "stateType", "nMeasurements", "volume_id", "layer_id"]])

print("\n\nStates with stateType=1 (Outlier) but sentinel measurement_id:")
outlier_type_states = track1_states[
    (track1_states.stateType == 1) & 
    (track1_states.measurement_id == UINT64_MAX)
]
print(outlier_type_states[["measurement_id", "stateType", "nMeasurements", "volume_id", "layer_id"]])

# Check if the CSV measurement IDs might correspond to outlier states that lost their source links
print("\n\nHypothesis: Maybe some measurements became outliers and lost source links?")
print("Let's check if we can find the CSV measurement IDs by looking at volume/layer info...")


States with stateType=0 (Measurement) but sentinel measurement_id:
Empty DataFrame
Columns: [measurement_id, stateType, nMeasurements, volume_id, layer_id]
Index: []


States with stateType=1 (Outlier) but sentinel measurement_id:
Empty DataFrame
Columns: [measurement_id, stateType, nMeasurements, volume_id, layer_id]
Index: []


Hypothesis: Maybe some measurements became outliers and lost source links?
Let's check if we can find the CSV measurement IDs by looking at volume/layer info...


In [35]:
# The issue: CSV writer uses MeasurementFlag, trackstates writer uses hasUncalibratedSourceLink()
# If source links are lost (e.g., during smoothing), states appear in CSV but not trackstates

# Let's check: Do we have exactly 5 states with MeasurementFlag but no source link?
# This would explain why CSV has 5 measurements but trackstates has 5 sentinel values

print("Summary:")
print(f"CSV says track 1 has {len(csv_ids)} measurements")
print(f"Trackstates has {len(track1_states[track1_states.measurement_id != UINT64_MAX])} states with valid measurement_id")
print(f"Trackstates has {len(track1_states[track1_states.measurement_id == UINT64_MAX])} states with sentinel measurement_id")

# Check if this is a version issue - maybe v1 vs v2 have different behavior?
print("\n\nLet's check v1 to see if it has the same issue...")


Summary:
CSV says track 1 has 11 measurements
Trackstates has 6 states with valid measurement_id
Trackstates has 5 states with sentinel measurement_id


Let's check v1 to see if it has the same issue...


In [36]:
# Compare v1 and v2 to see if this is a regression
# Reload v1 data (from earlier cells)
trackstates_v1 = "/global/cfs/cdirs/m4958/data/ColliderML/simulation/hard_scatter/ttbar/v1/runs/0/trackstates_ambi.root"
tracks_csv_v1 = "/global/cfs/cdirs/m4958/data/ColliderML/simulation/hard_scatter/ttbar/v1/runs/0/event000000000-tracks_ambi.csv"

track_states_df_v1 = load_root_file(str(trackstates_v1), included_columns=included_trackstates_columns, events=(0,1))
tracks_csv_v1_df = pd.read_csv(tracks_csv_v1)

print("=== V1 COMPARISON ===")
track1_states_v1 = track_states_df_v1[track_states_df_v1.track_nr == 1].copy()
csv_measurements_v1 = tracks_csv_v1_df[tracks_csv_v1_df.track_id == 1]["Measurements_ID"].iloc[0]
csv_ids_v1 = ast.literal_eval(csv_measurements_v1) if isinstance(csv_measurements_v1, str) else csv_measurements_v1

print(f"V1 CSV measurement IDs: {sorted(csv_ids_v1)}")
valid_measurement_ids_v1 = track1_states_v1[track1_states_v1.measurement_id != UINT64_MAX]["measurement_id"].tolist()
print(f"V1 Trackstates valid measurement IDs: {sorted(valid_measurement_ids_v1)}")
print(f"V1 Match: {set(csv_ids_v1) == set(valid_measurement_ids_v1)}")

print("\n=== V2 COMPARISON (from above) ===")
print(f"V2 CSV measurement IDs: {sorted(csv_ids)}")
print(f"V2 Trackstates valid measurement IDs: {sorted(valid_measurement_ids)}")
print(f"V2 Match: {set(csv_ids) == set(valid_measurement_ids)}")


=== V1 COMPARISON ===
V1 CSV measurement IDs: [145, 175, 306, 759, 867, 1079, 1109, 1552, 1553, 1574, 1576]


AttributeError: 'DataFrame' object has no attribute 'measurement_id'